# CellOracle BCL6 in silico KO on post-cardiomyocytes

This notebook is adapted from the official CellOracle KO simulation tutorial:
https://morris-lab.github.io/CellOracle.documentation/notebooks/05_simulation/Gata1_KO_simulation_with_Paul_etal_2015_data.html

It reuses the project's already-exported expression matrix and metadata
(produced by `scripts/01_export_from_seurat.R`) so we don't have to redo any
QC. The output of this notebook is a single AnnData file with two layers,
`imputed_count` (original) and `simulated_count` (post-BCL6-KO), which the
project's `scripts/07_run_celloracle_bcl6.py` then converts into the standard
genes x cells delta CSV.

Environment: install `envs/celloracle.yml` and select that kernel.

```bash
conda env create -f envs/celloracle.yml
conda activate bcl6-celloracle
python -m ipykernel install --user --name bcl6-celloracle
```

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import yaml

import celloracle as co
print('CellOracle version:', co.__version__)

In [ ]:
# Resolve project root and load shared config
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print('Working dir:', PROJECT_ROOT)

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

EXPR_PATH = Path(config['paths']['exported_expression_csv'])
META_PATH = Path(config['paths']['exported_meta_csv'])
NATIVE_OUT = Path(config['model_runners']['celloracle']['native_output_path'])
TARGET_GENE_CANDIDATES = config['perturbation']['target_gene_candidates']
KD_STRENGTH = float(config['perturbation']['knockdown_strength'])
CLUSTER_COL = config['metadata'].get('subtype_column', 'cm.subtype')
CONDITION_COL = config['metadata']['condition_column']

NATIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
print('Native output target:', NATIVE_OUT)

## 1. Build AnnData from the exported Seurat matrices

The exported matrix is genes x cells on Seurat's SCT scale (`scale.data`).
CellOracle expects an AnnData where rows are cells and columns are genes,
with raw-ish counts in `.X` for the imputation step. Because we do not have
raw counts re-exported, we run CellOracle on the SCT-scaled values; the
downstream simulate_shift returns a delta in the same scale, which is what
the project's evaluator expects.

In [ ]:
expr = pd.read_csv(EXPR_PATH, index_col=0)
expr.index = expr.index.astype(str)
expr.columns = expr.columns.astype(str)
meta = pd.read_csv(META_PATH, index_col=0)
if 'cell_id' in meta.columns:
    meta.index = meta['cell_id'].astype(str)
common = meta.index.intersection(expr.columns)
expr = expr.loc[:, common]
meta = meta.loc[common]
print('Expression shape (genes x cells):', expr.shape)
print('Metadata shape:', meta.shape)

In [ ]:
adata = ad.AnnData(
    X=expr.values.T.astype(np.float32),
    obs=meta.copy(),
    var=pd.DataFrame(index=expr.index.astype(str)),
)
adata.layers['imputed_count'] = adata.X.copy()
if CLUSTER_COL not in adata.obs.columns:
    print(f'Warning: cluster column {CLUSTER_COL!r} not found, using condition column instead.')
    adata.obs['_celloracle_cluster'] = adata.obs[CONDITION_COL].astype(str).values
    CLUSTER_COL = '_celloracle_cluster'
adata.obs[CLUSTER_COL] = adata.obs[CLUSTER_COL].astype('category')
adata

In [ ]:
# Standard scanpy embedding so CellOracle has a reduced space to project the
# perturbation through. If the Seurat object already has UMAP/PCA, you can
# skip this block and feed the precomputed coordinates instead.
sc.pp.pca(adata, n_comps=30)
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca')
sc.tl.umap(adata)
sc.pl.umap(adata, color=[CLUSTER_COL], show=False)

## 2. Initialise the Oracle and connect a base GRN

We use the human promoter base GRN bundled with CellOracle. If you have a
tissue-specific (e.g. cardiomyocyte ATAC-derived) base GRN, replace the
`base_GRN = co.data.load_human_promoter_base_GRN()` line with your own
DataFrame in the same format.

In [ ]:
oracle = co.Oracle()
oracle.import_anndata_as_normalized_count(
    adata=adata,
    cluster_column_name=CLUSTER_COL,
    embedding_name='X_umap',
)

base_GRN = co.data.load_human_promoter_base_GRN()
print('Base GRN shape:', base_GRN.shape)
oracle.import_TF_data(TF_info_matrix=base_GRN)

In [ ]:
# KNN imputation in PCA space — required before simulate_shift.
oracle.perform_PCA()
n_comps = min(50, oracle.adata.shape[1] - 1)
k = max(int(0.025 * oracle.adata.shape[0]), 25)
oracle.knn_imputation(
    n_pca_dims=n_comps,
    k=k,
    balanced=True,
    b_sight=k * 8,
    b_maxl=k * 4,
    n_jobs=4,
)
print('KNN imputation done. layers:', list(oracle.adata.layers.keys()))

## 3. Fit per-cluster GRN models

In [ ]:
links = oracle.get_links(
    cluster_name_for_GRN_unit=CLUSTER_COL,
    alpha=10,
    verbose_level=1,
)
links.filter_links(p=0.001, weight='coef_abs', threshold_number=2000)
links.get_network_score()
oracle.get_cluster_specific_TFdict_from_Links(links_object=links)
oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)
print('GRN fit done.')

## 4. BCL6 in silico knockdown

In [ ]:
target_gene = next((g for g in TARGET_GENE_CANDIDATES if g in oracle.adata.var_names), None)
if target_gene is None:
    raise ValueError(f'None of {TARGET_GENE_CANDIDATES} found in oracle.adata.var_names')
print('Target gene resolved to:', target_gene)

# CellOracle interprets the perturb_condition value as the new (clamped) expression.
# Setting it to 0 simulates a hard KO; multiplying current mean by (1 - strength)
# simulates a partial knockdown.
current_mean = float(oracle.adata[:, target_gene].X.mean())
perturbed_value = current_mean * (1.0 - KD_STRENGTH)
print(f'Original mean expression of {target_gene}: {current_mean:.4f}')
print(f'Perturbed clamped value (knockdown strength={KD_STRENGTH}): {perturbed_value:.4f}')

oracle.simulate_shift(
    perturb_condition={target_gene: perturbed_value},
    n_propagation=3,
)
print('Simulation done. layers now in oracle.adata:', list(oracle.adata.layers.keys()))

## 5. Save the native artefact for the project converter

`scripts/07_run_celloracle_bcl6.py` reads this `.h5ad` and writes the
delta-expression CSV that `scripts/05_compare_perturbation_models.py` expects.

In [ ]:
ad_out = ad.AnnData(
    X=np.zeros_like(oracle.adata.layers['imputed_count']),
    obs=oracle.adata.obs.copy(),
    var=oracle.adata.var.copy(),
)
ad_out.layers['imputed_count'] = np.asarray(oracle.adata.layers['imputed_count'])
ad_out.layers['simulated_count'] = np.asarray(oracle.adata.layers['simulated_count'])
ad_out.uns['perturbation'] = {
    'target_gene': target_gene,
    'knockdown_strength': KD_STRENGTH,
    'perturbed_value': perturbed_value,
    'cluster_column': CLUSTER_COL,
}
ad_out.write_h5ad(NATIVE_OUT)
print('Wrote native CellOracle artefact to', NATIVE_OUT)
print('Now run: python scripts/07_run_celloracle_bcl6.py --config configs/config.yaml')